# 投机解码--EAGLE-3

## 问题描述

推理时只产生一个词元，瓶颈在带宽。投机解码的理念是使用一个便宜的草案模型N次前向生成N个词元，然后在验证模型中一次前向验证。如果验证模型在第i个词元处的分布概率满足，接受这个词，否则拒绝掉。最好情况下，N个词元都被接受，验证模型额外产生下一个词元，即一次前向吐出N+1个词元。

## 基本概念

### 核心：Leviathan 拒绝采样

假设`p(t)`是草案模型基于上下文对下一个词元的预测，假设`q(t)`是验证模型的。对于一个词元提案`d~p`，有`min(1, q(d) / p(d))` 的概率接受。拒绝的时候，从残差分布上进行采样`(q - p)_ + ||(q - p)_+||_1`。结果采样服从`q`的分布。

不管`p`有多差，越差拒绝的频率越高，但是输出仍然保持精确。

这就是EAGLE 无损推理加速的核心理念。

### 决定推理加速

假设`a`是每个词元提案的预期接受率，假设`c=cost(draft)/cost(verifer)`是开销比率，验证模型一次前向预期的接受词元数为：
```
E[accepted] = (1 - a^(N+1)) / (1 - a)
```
总耗时为
```
Cost = N * c + 1

Cost_per_token = Cost / E[accepted]
```